In [6]:
# =========================
# Standard library imports
# =========================
import sys
import os
import io
import gzip
from pathlib import Path
import math
import json
import pickle
import logging
import random
import subprocess
import warnings
import collections
import itertools
import re
from IPython.display import display

warnings.filterwarnings("ignore")


# =========================
# Numeric / stats
# =========================
import numpy as np
import pandas as pd
from scipy import sparse
from scipy import stats
from scipy.stats import rankdata
from scipy import __version__ as scipy_version


# =========================
# Plotting / visualization
# =========================
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, FormatStrFormatter, MaxNLocator
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
import seaborn as sns


# =========================
# scikit-learn
# =========================
from sklearn import model_selection, metrics
from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit,
    KFold,
    GroupKFold,
    cross_val_score,
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neighbors import LocalOutlierFactor
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
)
from sklearn import __version__ as sklearn_version


# =========================
# Optimization / AutoML
# =========================
import optuna


# =========================
# LightGBM
# =========================
try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
    _HAS_LGBM = True
except Exception as e:
    _HAS_LGBM = False
    raise RuntimeError(
        "LightGBM not installed. Please install it with `pip install lightgbm`."
    ) from e


# =========================
# RNA structure (ViennaRNA)
# =========================
import RNA


# =========================
# UpSet plots
# =========================
try:
    from upsetplot import UpSet, from_memberships
except Exception:
    _ = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "upsetplot"],
        check=False,
    )
    from upsetplot import UpSet, from_memberships


# =========================
# Model explanation
# =========================
import shap


# =========================
# User-provided utilities
# =========================
import sylib  


# =========================
# Version info
# =========================
print(f"python    = {sys.version_info[0]}.{sys.version_info[1]}.{sys.version_info[2]}")
print(f"pandas    = {pd.__version__}")
print(f"numpy     = {np.__version__}")
print(f"scipy     = {scipy_version}")
print(f"optuna    = {optuna.__version__}")
print(f"sklearn   = {sklearn_version}")
print(f"ViennaRNA = {RNA.__version__}")
print(f"lightgbm  = {lgb.__version__}")
print(f"sylib     = {sylib.__version__}")


# =========================
# Progress bar & logging
# =========================
# progress bar from sylib
progress_bar = sylib.utils.ProgressBar()

# reset handlers then configure logging
logging.root.handlers = []
stream_handler = logging.StreamHandler(sys.stderr)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)8s: %(message)s",
    handlers=[stream_handler],
)
logger = logging.getLogger(__name__)

# make matplotlib quieter
logging.getLogger("matplotlib").setLevel(logging.WARNING)

python    = 3.11.15
pandas    = 2.3.3
numpy     = 2.4.6
scipy     = 1.17.1
optuna    = 4.9.0
sklearn   = 1.9.0
ViennaRNA = 2.7.2
lightgbm  = 4.6.0
sylib     = 0.3.0.dev0+ae18bb2


In [7]:
# ============================================
# Global plotting configuration
# ============================================

_PLOT_CFG = {
    "fig_w": 6.0,
    "fig_h": 6.0,
    "dpi": 300,
}


SPECIES_INFO = {
    "AT21": {
        "label": "AT",
        "short": "AT21",
        "color": "#664D0AFF",
        "marker": "o",
    },
    "NB21": {
        "label": "NB",
        "short": "NB21",
        "color": "#7e3131",
        "marker": "^",
    },
    "OS21": {
        "label": "OS",
        "short": "OS21",
        "color": "#13563f",
        "marker": "s",
    },
}

def set_plot_style(
    *,
    base_fontsize=10,
    title_fontsize=12,
    label_fontsize=10,
    tick_fontsize=9,
    legend_fontsize=10,
    dpi=300,
    axes_linewidth=1.2,
    spines_top=True,
    spines_right=True,
    tick_size_major=6,
    tick_dir="out",
    grid=False,
    fig_w=6.0,
    fig_h=6.0,
):
    sns.set_style("ticks")

    mpl.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": base_fontsize,

        "axes.titlesize": title_fontsize,
        "axes.labelsize": label_fontsize,

        "xtick.labelsize": tick_fontsize,
        "ytick.labelsize": tick_fontsize,

        "legend.fontsize": legend_fontsize,

        "figure.dpi": dpi,
        "savefig.dpi": dpi,

        "axes.linewidth": axes_linewidth,
        "axes.spines.top": spines_top,
        "axes.spines.right": spines_right,
        "axes.grid": grid,
        "axes.axisbelow": True,

        "xtick.major.size": tick_size_major,
        "ytick.major.size": tick_size_major,
        "xtick.direction": tick_dir,
        "ytick.direction": tick_dir,

        "legend.frameon": False,

        "savefig.bbox": "tight",
        "savefig.transparent": False,
        "figure.autolayout": False,
    })

    _PLOT_CFG.update({
        "fig_w": fig_w,
        "fig_h": fig_h,
        "dpi": dpi,
    })


def make_fig(w=None, h=None, dpi=None):
    W = float(w) if w is not None else _PLOT_CFG["fig_w"]
    H = float(h) if h is not None else _PLOT_CFG["fig_h"]
    D = dpi if dpi is not None else _PLOT_CFG["dpi"]

    fig, ax = plt.subplots(
        figsize=(W, H),
        dpi=D,
    )

    return fig, ax


def _compact_formatter():
    def _fmt(x, _pos=None):
        axx = abs(x)

        if axx >= 1e9:
            s = f"{x / 1e9:.1f}B"
        elif axx >= 1e6:
            s = f"{x / 1e6:.1f}M"
        elif axx >= 1e3:
            s = f"{x / 1e3:.1f}k"
        else:
            s = f"{x:.2g}"

        return (
            s.replace(".0B", "B")
             .replace(".0M", "M")
             .replace(".0k", "k")
        )

    return FuncFormatter(_fmt)


def format_axis(
    ax,
    *,
    xlabel=None,
    ylabel=None,
    compact_ticks=(),
):
    if xlabel is not None:
        ax.set_xlabel(xlabel)

    if ylabel is not None:
        ax.set_ylabel(ylabel)

    fmt = _compact_formatter()

    if "x" in compact_ticks:
        ax.xaxis.set_major_formatter(fmt)

    if "y" in compact_ticks:
        ax.yaxis.set_major_formatter(fmt)

    return ax


# ============================================
# Joint scatter with KDE marginals
# ============================================

def safe_pearsonr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) < 2:
        return np.nan, np.nan

    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan

    return stats.pearsonr(x, y)


def safe_spearmanr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) < 2:
        return np.nan, np.nan

    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan

    return stats.spearmanr(x, y)

def _kde_1d(values, lo, hi, num=256):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    grid = np.linspace(lo, hi, num)

    if len(values) < 2:
        return grid, np.zeros_like(grid)

    try:
        kde = stats.gaussian_kde(values)
        dens = kde(grid)
        dens /= dens.max() if dens.max() > 0 else 1
        return grid, dens

    except Exception:
        return grid, np.zeros_like(grid)


def joint_scatter(
    x,
    y,
    *,
    color=None,
    point_size=18,
    alpha=0.65,
    show_identity=True,
    show_regression=True,
    annotate=True,
    annotate_spearman=True,
    xlabel=None,
    ylabel=None,
    title=None,
    figsize=None,
    w=None,
    h=None,
    dpi=None,
    annotate_fontsize=14,
):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    n = len(x)

    if figsize is not None:
        FW, FH = figsize
    else:
        FW = float(w) if w is not None else _PLOT_CFG["fig_w"]
        FH = float(h) if h is not None else _PLOT_CFG["fig_h"]

    fig = plt.figure(
        figsize=(FW, FH),
        dpi=(dpi or _PLOT_CFG["dpi"]),
    )

    gs = GridSpec(
        2,
        2,
        width_ratios=(4, 1),
        height_ratios=(1, 4),
        hspace=0.05,
        wspace=0.05,
    )

    ax_top = fig.add_subplot(gs[0, 0])
    ax_joint = fig.add_subplot(gs[1, 0], sharex=ax_top)
    ax_right = fig.add_subplot(gs[1, 1], sharey=ax_joint)

    ax_joint.scatter(
        x,
        y,
        s=point_size,
        alpha=alpha,
        edgecolor="none",
        color=color,
    )

    lo = float(np.nanmin([x.min(), y.min()]))
    hi = float(np.nanmax([x.max(), y.max()]))

    pad = 0.05 * (hi - lo if hi > lo else 1.0)

    lo -= pad
    hi += pad

    ax_joint.set_xlim(lo, hi)
    ax_joint.set_ylim(lo, hi)

    if show_identity:
        ax_joint.plot(
            [lo, hi],
            [lo, hi],
            ls="--",
            lw=1.2,
            color="0.65",
            zorder=1,
        )

    if show_regression and n >= 2 and np.std(x) > 0 and np.std(y) > 0:
        slope, intercept = np.polyfit(x, y, 1)

        ax_joint.plot(
            [lo, hi],
            slope * np.array([lo, hi]) + intercept,
            color="black",
            lw=1.5,
            zorder=2,
        )

    format_axis(
        ax_joint,
        xlabel=xlabel,
        ylabel=ylabel,
        compact_ticks=(),
    )

    if title:
        ax_joint.set_title(title)

    if annotate and n >= 2 and np.std(x) > 0 and np.std(y) > 0:
        rp, _ = safe_pearsonr(x, y)
        rs, _ = safe_spearmanr(x, y)

        txt = rf"$r_p = {rp:.2f}$"

        if annotate_spearman:
            txt += "\n" + rf"$r_s = {rs:.2f}$"

        txt += f"\n$n = {n}$"

        ax_joint.text(
            0.04,
            0.96,
            txt,
            transform=ax_joint.transAxes,
            ha="left",
            va="top",
            fontsize=annotate_fontsize,
        )

    gx, dx = _kde_1d(x, lo, hi)
    gy, dy = _kde_1d(y, lo, hi)

    ax_top.plot(gx, dx, lw=2, color=color)
    ax_top.axis("off")

    ax_right.plot(dy, gy, lw=2, color=color)
    ax_right.axis("off")

    plt.tight_layout()

    return fig, (ax_joint, ax_top, ax_right)


set_plot_style()

# ============================================
# Plot helper functions
# ============================================

def species_palette():
    return [
        SPECIES_INFO["AT21"]["color"],
        SPECIES_INFO["NB21"]["color"],
        SPECIES_INFO["OS21"]["color"],
    ]


def outside_legend(
    ax,
    *,
    title="Species",
    n_items=3,
    x=1.02,
    y=1.00,
):
    handles, labels = ax.get_legend_handles_labels()

    ax.legend(
        handles[:n_items],
        labels[:n_items],
        title=title,
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(x, y),
        borderaxespad=0,
    )

    return ax

In [8]:
# ============================================
# Paths
# ============================================

RESULT_DIR = Path(
    "/mnt/d/Ibnu/Programming/data/regression/results/lgbm"
)

SPECIES = [
    "AT21",
    "NB21",
    "OS21",
]

# --------------------------------------------
# Candidate table
# --------------------------------------------

candidate_file = (
    RESULT_DIR
    / (
        "lgbm.species_specific_candidate_table"
        ".importance_p90.sss_p90.tsv.gz"
    )
)

candidate_all_df = pd.read_csv(
    candidate_file,
    sep="\t",
)

candidate_selected_df = (
    candidate_all_df.loc[
        candidate_all_df["Selected"]
    ]
    .copy()
    .reset_index(drop=True)
)

print("Candidate table")
print("----------------")
print("All rows :", len(candidate_all_df))
print("Selected :", len(candidate_selected_df))

display(
    candidate_selected_df.head()
)


# --------------------------------------------
# Load species-specific LightGBM packages
# --------------------------------------------

lgbm_models = {}
lgbm_data = {}

for species in SPECIES:

    model_file = (
        RESULT_DIR
        / f"{species}.lgbm.model.txt"
    )

    bundle_file = (
        RESULT_DIR
        / f"{species}.lgbm.data_bundle.pkl.gz"
    )

    # fitted LightGBM model
    model = lgb.Booster(
        model_file=str(model_file)
    )

    # reusable model data
    with gzip.open(
        bundle_file,
        "rb",
    ) as f:
        bundle = pickle.load(f)

    # ----------------------------------------
    # Integrity checks
    # ----------------------------------------

    assert bundle["species"] == species

    assert (
        bundle["feature_names"]
        ==
        list(bundle["x_train_model"].columns)
    )

    assert (
        bundle["feature_names"]
        ==
        list(bundle["x_test_model"].columns)
    )

    assert (
        bundle["feature_names"]
        ==
        list(bundle["x_train_raw"].columns)
    )

    assert (
        bundle["feature_names"]
        ==
        list(bundle["x_test_raw"].columns)
    )

    lgbm_models[species] = model
    lgbm_data[species] = bundle

    print()
    print("=" * 60)
    print(species)
    print("Train :", bundle["x_train_model"].shape)
    print("Test  :", bundle["x_test_model"].shape)
    print("Features:", len(bundle["feature_names"]))


print()
print("All reusable LightGBM packages loaded successfully.")

Candidate table
----------------
All rows : 1347
Selected : 70


,Species,Feature,Region,Feature_type,Importance_raw,Importance_normalized,Importance_percentile,SSS,SSS_percentile,Selected
0,AT,5'UTR.Length,5'UTR,Length,21.0,0.005526,0.974388,0.010997,0.997773,True
1,AT,mRNA.MFE,mRNA,RNA_structure_MFE,19.0,0.005000,0.958797,0.008363,0.993318,True
2,AT,3'UTR.UAG-freq,3'UTR,Nucleotide_kmer_freq,21.0,0.005526,0.974388,0.006610,0.986637,True
3,AT,5'UTR.U-freq,5'UTR,Nucleotide_kmer_freq,26.0,0.006842,0.993318,0.006238,0.979955,True
4,AT,CDS.UAU-freq,CDS,Nucleotide_kmer_freq,17.0,0.004474,0.927617,0.005743,0.973274,True



AT21
Train : (5698, 464)
Test  : (1428, 464)
Features: 464

NB21
Train : (4305, 464)
Test  : (1076, 464)
Features: 464

OS21
Train : (4445, 463)
Test  : (1107, 463)
Features: 463

All reusable LightGBM packages loaded successfully.


In [9]:
# ============================================
# Check candidate features against
# the three LightGBM feature spaces
# ============================================

# --------------------------------------------
# Feature sets for each fitted model
# --------------------------------------------

model_feature_sets = {
    species: set(
        lgbm_data[species]["feature_names"]
    )
    for species in SPECIES
}


# --------------------------------------------
# Selected candidate features
# --------------------------------------------

selected_features = set(
    candidate_selected_df["Feature"]
)

print(
    "Selected candidate rows   :",
    len(candidate_selected_df),
)

print(
    "Unique selected features  :",
    len(selected_features),
)


# --------------------------------------------
# Check selected features in each model
# --------------------------------------------

rows = []

for feature in sorted(selected_features):

    rows.append({
        "Feature": feature,
        "AT21": feature in model_feature_sets["AT21"],
        "NB21": feature in model_feature_sets["NB21"],
        "OS21": feature in model_feature_sets["OS21"],
    })

candidate_feature_check_df = pd.DataFrame(rows)

candidate_feature_check_df["Present_all_species"] = (
    candidate_feature_check_df[
        [
            "AT21",
            "NB21",
            "OS21",
        ]
    ]
    .all(axis=1)
)

display(
    candidate_feature_check_df.head()
)


# --------------------------------------------
# Summary
# --------------------------------------------

print()
print("Candidate feature availability")
print("------------------------------")

print(
    "Present in AT21:",
    candidate_feature_check_df["AT21"].sum(),
)

print(
    "Present in NB21:",
    candidate_feature_check_df["NB21"].sum(),
)

print(
    "Present in OS21:",
    candidate_feature_check_df["OS21"].sum(),
)

print(
    "Present in all three:",
    candidate_feature_check_df[
        "Present_all_species"
    ].sum(),
)


# --------------------------------------------
# Show any missing candidate features
# --------------------------------------------

missing_candidate_df = (
    candidate_feature_check_df.loc[
        ~candidate_feature_check_df[
            "Present_all_species"
        ]
    ]
    .copy()
)

print()
print(
    "Candidate features missing from"
    " at least one model:",
    len(missing_candidate_df),
)

display(
    missing_candidate_df
)

Selected candidate rows   : 70
Unique selected features  : 61


,Feature,AT21,NB21,OS21,Present_all_species
0,3'UTR.ACA-freq,True,True,True,True
1,3'UTR.AG-freq,True,True,True,True
2,3'UTR.AUC-freq,True,True,True,True
3,3'UTR.G-freq,True,True,True,True
4,3'UTR.GAU-freq,True,True,True,True



Candidate feature availability
------------------------------
Present in AT21: 61
Present in NB21: 61
Present in OS21: 61
Present in all three: 61

Candidate features missing from at least one model: 0


,Feature,AT21,NB21,OS21,Present_all_species


In [12]:
# ============================================
# Build raw candidate-feature dataset
# using saved LightGBM data bundles
# ============================================

candidate_features = sorted(candidate_selected_df["Feature"].unique())

print(f"Unique candidate features: {len(candidate_features)}")

raw_candidate_wide = {}

for species in SPECIES:

    bundle = lgbm_data[species]

    # ----------------------------------------
    # Combine raw feature values
    # ----------------------------------------

    x_train_raw = bundle["x_train_raw"].loc[:, candidate_features].copy()
    x_test_raw = bundle["x_test_raw"].loc[:, candidate_features].copy()

    x_raw = pd.concat([x_train_raw, x_test_raw], axis=0)

    # ----------------------------------------
    # Combine metadata
    # ----------------------------------------

    train_meta = bundle["train_metadata"].copy()
    test_meta = bundle["test_metadata"].copy()

    metadata = pd.concat([train_meta, test_meta], axis=0)

    # ----------------------------------------
    # Integrity check
    # ----------------------------------------

    assert x_raw.index.equals(metadata.index)

    # ----------------------------------------
    # Build species dataframe
    # ----------------------------------------

    raw_df = pd.concat([metadata, x_raw], axis=1)

    raw_df.insert(0, "Species", SPECIES_INFO[species]["label"])
    raw_df.insert(1, "Species_key", species)

    raw_candidate_wide[species] = raw_df

# ============================================
# Combine all species
# ============================================

candidate_raw_wide_df = pd.concat(
    raw_candidate_wide.values(),
    ignore_index=True,
)

print("\nCombined raw candidate-feature dataset")
print("--------------------------------------")
print(f"Shape: {candidate_raw_wide_df.shape}")

print("\nSpecies counts")
display(candidate_raw_wide_df["Species"].value_counts())
print(candidate_raw_wide_df.head())
# ============================================
# Descriptive summary of raw candidate features
# by species
# ============================================

candidate_feature_summary_df = (
    candidate_raw_wide_df
    .groupby("Species")[candidate_features]
    .agg(["count", "mean", "std", "median", "min", "max"])
)

display(candidate_feature_summary_df)

Unique candidate features: 61

Combined raw candidate-feature dataset
--------------------------------------
Shape: (18059, 70)

Species counts


Species
AT    7126
OS    5552
NB    5381
Name: count, dtype: int64

  Species Species_key                         var_id     trans_id    gene_id  \
0      AT        AT21    AT5G05370.1.1591901.1590815  AT5G05370.1  AT5G05370   
1      AT        AT21    AT5G16060.1.5246121.5247413  AT5G16060.1  AT5G16060   
2      AT        AT21  AT2G34160.1.14426246.14427367  AT2G34160.1  AT2G34160   
3      AT        AT21  AT5G54600.1.22183004.22184509  AT5G54600.1  AT5G54600   
4      AT        AT21    AT2G23340.1.9937988.9938873  AT2G23340.1  AT2G23340   

    dataset  observed_raw  observed_model  predicted_model  3'UTR.ACA-freq  \
0  Training      0.632897        0.054841         0.418203        6.172840   
1  Training      0.365525       -1.094430        -0.499050        9.803922   
2  Training      0.600538       -0.073162        -0.132563       13.605442   
3  Training      0.544256       -0.302488        -0.542934        0.000000   
4  Training      1.047809        1.491571         0.360980       19.108280   

   ...  mRNA.GCU-freq  mRNA.GU-freq  mRNA.GUA-freq

3'UTR.ACA-freq                                                  \
                 count       mean       std    median  min         max   
Species                                                                  
AT                7126  11.284752  9.932164  9.708738  0.0   68.965517   
NB                5381  10.003368  8.471785  8.620690  0.0  104.166667   
OS                5552  11.296883  8.823811  9.803922  0.0   52.884615   

        3'UTR.AG-freq                                   ... mRNA.UC-freq  \
                count       mean        std     median  ...          std   
Species                                                 ...                
AT               7126  43.455944  18.092570  42.372881  ...    14.261474   
NB               5381  47.933791  17.074809  47.904192  ...    11.443883   
OS               5552  49.043126  16.686696  48.913043  ...    13.257842   

                                          mRNA.Y-freq                         \
            median        min         max       count        mean        std   
Species                                                                        
AT       72.139303  26.178010  155.102041        7126  493.954042  36.211605   
NB       59.625213  27.164686  110.619469        5381  489.981309  34.913898   
OS       68.435754  18.261965  132.231405        5552  502.223150  32.356568   

                                             
             median         min         max  
Species                                      
AT       493.281067  332.425068  672.881356  
NB       489.169675  362.318841  627.952756  
OS       503.089295  377.740304  624.772313  

[3 rows x 366 columns]

In [13]:
# ============================================
# Long-format candidate feature summary
# ============================================

summary_rows = []

for species in ["AT", "NB", "OS"]:

    df = candidate_raw_wide_df[
        candidate_raw_wide_df["Species"] == species
    ]

    for feature in candidate_features:

        values = df[feature].dropna()

        summary_rows.append({
            "Species": species,
            "Feature": feature,
            "n": len(values),
            "mean": values.mean(),
            "std": values.std(),
            "median": values.median(),
            "q25": values.quantile(0.25),
            "q75": values.quantile(0.75),
            "min": values.min(),
            "max": values.max(),
        })

candidate_feature_summary_long_df = pd.DataFrame(summary_rows)

candidate_feature_summary_long_df = (
    candidate_feature_summary_long_df
    .merge(
        candidate_selected_df[
            ["Feature", "Region", "Feature_type"]
        ]
        .drop_duplicates(),
        on="Feature",
        how="left",
    )
)

display(candidate_feature_summary_long_df.head())

,Species,Feature,n,mean,std,median,q25,q75,min,max,Region,Feature_type
0,AT,3'UTR.ACA-freq,7126,11.284752,9.932164,9.708738,4.830918,16.393443,0.00000,68.965517,3'UTR,Nucleotide_kmer_freq
1,AT,3'UTR.AG-freq,7126,43.455944,18.092570,42.372881,31.421845,54.263566,0.00000,148.148148,3'UTR,Nucleotide_kmer_freq
2,AT,3'UTR.AUC-freq,7126,18.525699,11.346800,17.094017,10.309278,25.196871,0.00000,105.263158,3'UTR,Nucleotide_kmer_freq
3,AT,3'UTR.G-freq,7126,172.249670,34.196736,172.774869,150.326797,193.798450,21.73913,363.636364,3'UTR,Nucleotide_kmer_freq
4,AT,3'UTR.GAU-freq,7126,19.107572,11.747014,18.072289,10.989011,25.613748,0.00000,128.205128,3'UTR,Nucleotide_kmer_freq


In [23]:
# ============================================
# Spearman correlation among candidate features
# by species
# ============================================

candidate_corr = {}

for species in ["AT", "NB", "OS"]:

    df = candidate_raw_wide_df[
        candidate_raw_wide_df["Species"] == species
    ]

    corr = df[candidate_features].corr(
        method="spearman"
    )

    candidate_corr[species] = corr

    print("=" * 60)
    print(species)
    print("shape:", corr.shape)

    display(corr.iloc[:8, :8])

AT
shape: (61, 61)


,3'UTR.ACA-freq,3'UTR.AG-freq,3'UTR.AUC-freq,3'UTR.G-freq,3'UTR.GAU-freq,3'UTR.GUA-freq,3'UTR.Length,3'UTR.UAG-freq
3'UTR.ACA-freq,1.000000,0.123291,0.003866,-0.210627,-0.119391,0.034597,0.007159,-0.065685
3'UTR.AG-freq,0.123291,1.000000,-0.089267,0.343586,0.046116,0.057656,0.245684,0.411924
3'UTR.AUC-freq,0.003866,-0.089267,1.000000,-0.249175,0.073869,0.021216,-0.008747,-0.018959
3'UTR.G-freq,-0.210627,0.343586,-0.249175,1.000000,0.252996,0.014374,0.285739,0.169505
3'UTR.GAU-freq,-0.119391,0.046116,0.073869,0.252996,1.000000,-0.141035,0.009478,0.113653
3'UTR.GUA-freq,0.034597,0.057656,0.021216,0.014374,-0.141035,1.000000,-0.059060,0.109127
3'UTR.Length,0.007159,0.245684,-0.008747,0.285739,0.009478,-0.059060,1.000000,0.106260
3'UTR.UAG-freq,-0.065685,0.411924,-0.018959,0.169505,0.113653,0.109127,0.106260,1.000000


NB
shape: (61, 61)


,3'UTR.ACA-freq,3'UTR.AG-freq,3'UTR.AUC-freq,3'UTR.G-freq,3'UTR.GAU-freq,3'UTR.GUA-freq,3'UTR.Length,3'UTR.UAG-freq
3'UTR.ACA-freq,1.000000,0.074243,0.147105,-0.142624,-0.112511,0.014228,0.040089,-0.040143
3'UTR.AG-freq,0.074243,1.000000,-0.039742,0.424184,-0.001160,0.025643,0.219316,0.469455
3'UTR.AUC-freq,0.147105,-0.039742,1.000000,-0.160695,0.058427,0.012772,0.057414,0.011906
3'UTR.G-freq,-0.142624,0.424184,-0.160695,1.000000,0.186992,-0.005696,0.292446,0.225714
3'UTR.GAU-freq,-0.112511,-0.001160,0.058427,0.186992,1.000000,-0.097868,0.023898,0.014313
3'UTR.GUA-freq,0.014228,0.025643,0.012772,-0.005696,-0.097868,1.000000,-0.050710,0.065132
3'UTR.Length,0.040089,0.219316,0.057414,0.292446,0.023898,-0.050710,1.000000,0.216449
3'UTR.UAG-freq,-0.040143,0.469455,0.011906,0.225714,0.014313,0.065132,0.216449,1.000000


OS
shape: (61, 61)


,3'UTR.ACA-freq,3'UTR.AG-freq,3'UTR.AUC-freq,3'UTR.G-freq,3'UTR.GAU-freq,3'UTR.GUA-freq,3'UTR.Length,3'UTR.UAG-freq
3'UTR.ACA-freq,1.000000,0.025452,0.015587,-0.272732,-0.088722,0.032360,0.000786,-0.126830
3'UTR.AG-freq,0.025452,1.000000,-0.171084,0.295827,-0.003433,0.027417,0.108509,0.431710
3'UTR.AUC-freq,0.015587,-0.171084,1.000000,-0.280111,0.124518,0.050075,0.002471,-0.035988
3'UTR.G-freq,-0.272732,0.295827,-0.280111,1.000000,0.215621,-0.003055,0.206374,0.129246
3'UTR.GAU-freq,-0.088722,-0.003433,0.124518,0.215621,1.000000,-0.048924,0.006376,0.016997
3'UTR.GUA-freq,0.032360,0.027417,0.050075,-0.003055,-0.048924,1.000000,0.023432,0.180234
3'UTR.Length,0.000786,0.108509,0.002471,0.206374,0.006376,0.023432,1.000000,0.014713
3'UTR.UAG-freq,-0.126830,0.431710,-0.035988,0.129246,0.016997,0.180234,0.014713,1.000000


In [24]:
# ============================================
# Correlation summary
# ============================================

corr_summary_rows = []

for species, corr in candidate_corr.items():

    upper = corr.values[
        np.triu_indices_from(
            corr.values,
            k=1,
        )
    ]

    corr_summary_rows.append({
        "Species": species,
        "n_pairs": len(upper),
        "mean_abs_r": np.mean(np.abs(upper)),
        "median_abs_r": np.median(np.abs(upper)),
        "max_abs_r": np.max(np.abs(upper)),
        "n_abs_r_ge_05": np.sum(np.abs(upper) >= 0.50),
        "n_abs_r_ge_07": np.sum(np.abs(upper) >= 0.70),
        "n_abs_r_ge_09": np.sum(np.abs(upper) >= 0.90),
    })

corr_summary_df = pd.DataFrame(
    corr_summary_rows
)

display(corr_summary_df)

,Species,n_pairs,mean_abs_r,median_abs_r,max_abs_r,n_abs_r_ge_05,n_abs_r_ge_07,n_abs_r_ge_09
0,AT,1830,0.105676,0.062853,1.0,37,13,6
1,NB,1830,0.104230,0.066672,1.0,43,13,2
2,OS,1830,0.127821,0.084094,1.0,70,20,4


In [25]:
# ============================================
# Strong correlation overlap among species
# ============================================

CORR_THRESHOLD = 0.70

strong_pair_sets = {}

for species, corr in candidate_corr.items():

    pairs = set()

    for i, f1 in enumerate(candidate_features):

        for j in range(i + 1, len(candidate_features)):

            f2 = candidate_features[j]

            r = corr.loc[f1, f2]

            if abs(r) >= CORR_THRESHOLD:

                pairs.add(
                    tuple(sorted((f1, f2)))
                )

    strong_pair_sets[species] = pairs


print("Strong correlations")
print("-------------------")

for species in ["AT", "NB", "OS"]:

    print(
        species,
        len(strong_pair_sets[species])
    )


print()
print("Shared pairs")

print(
    "AT ∩ NB:",
    len(
        strong_pair_sets["AT"]
        &
        strong_pair_sets["NB"]
    )
)

print(
    "AT ∩ OS:",
    len(
        strong_pair_sets["AT"]
        &
        strong_pair_sets["OS"]
    )
)

print(
    "NB ∩ OS:",
    len(
        strong_pair_sets["NB"]
        &
        strong_pair_sets["OS"]
    )
)

shared_all = (
    strong_pair_sets["AT"]
    &
    strong_pair_sets["NB"]
    &
    strong_pair_sets["OS"]
)

print()

print(
    "Shared in all three:",
    len(shared_all)
)

Strong correlations
-------------------
AT 13
NB 13
OS 20

Shared pairs
AT ∩ NB: 12
AT ∩ OS: 11
NB ∩ OS: 11

Shared in all three: 11
